In [ ]:

from pathlib import Path
import sys

ROOT = Path.cwd().parent   # if notebook is inside notebooks/

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
from src.moire.extraction_helpers import local_poly, adaptive_smooth

IN = ROOT / Path("source_data")
print(IN)

def load_field(E, IN):
    df = pd.read_csv(IN / f'Rxx_matrix_E-{E}mV_nm.csv')

    T  = df.iloc[:, 0].astype(float).to_numpy()
    nu = np.array([float(c) for c in df.columns[1:]])
    R  = df.iloc[:, 1:].astype(float).to_numpy()

    # # Keep only rows where T is finite and the whole rho linecut is finite
    # mask = np.isfinite(T) & np.all(np.isfinite(R), axis=1)
    # T, R = T[mask], R[mask]

    # Sort rows by increasing temperature; R rows must follow T
    # idx = np.argsort(T)
    # T, R = T[idx], R[idx]

    return T, nu, R


T, nu, R = load_field(96.2, IN)


# Plotting regular field, without sorting

nu_interest = np.argmin(np.abs(nu - 0.796))
R_interest = R[:, nu_interest]

plt.plot(T, R_interest, alpha = 0.8)
plt.xlim = 0
plt.ylim = 0

print(T[-1])
print(T[0])
print(np.max(T))
print(np.min(T))
h_max = (T[-1] - T[0]) * 0.2
h_max = 0.05

# Also adding rough data 

rough = np.array([local_poly(R_interest, T, t, h_max, 1) for t in T])
plt.plot(T, rough, color = "red", alpha = 0.8)

# Also adding adaptive smoothed data

# adaptive_smoothed = adaptive_smooth(R_interest, T)
# plt.plot(T, adaptive_smoothed, color = "green", alpha = 0.8)



In [ ]:
def load_field_sorted(E, IN):
    df = pd.read_csv(IN / f'Rxx_matrix_E-{E}mV_nm.csv')

    T  = df.iloc[:, 0].astype(float).to_numpy()
    nu = np.array([float(c) for c in df.columns[1:]])
    R  = df.iloc[:, 1:].astype(float).to_numpy()

    # Keep only rows where T is finite and the whole rho linecut is finite
    mask = np.isfinite(T) & np.all(np.isfinite(R), axis=1)
    T, R = T[mask], R[mask]

    # Sort rows by increasing temperature; R rows must follow T
    idx = np.argsort(T)
    T, R = T[idx], R[idx]

    return T, nu, R


T_sorted, nu_sorted, R_sorted = load_field_sorted(96.2, IN)


# Plotting regular field, without sorting

nu_interest_sorted = np.argmin(np.abs(nu - 0.796))
R_interest_sorted = R[:, nu_interest_sorted]

plt.plot(T, R_interest_sorted)
plt.xlim = 0
plt.ylim = 0

In [ ]:

# Test how fallback smoothgin, with local_poly -> adaptive smoothing performs on ALL data

def load_field(E, IN):
    df = pd.read_csv(IN / f'Rxx_matrix_E-{E}mV_nm.csv')

    T  = df.iloc[:, 0].astype(float).to_numpy()
    nu = np.array([float(c) for c in df.columns[1:]])
    R  = df.iloc[:, 1:].astype(float).to_numpy()

    # # Keep only rows where T is finite and the whole rho linecut is finite
    # mask = np.isfinite(T) & np.all(np.isfinite(R), axis=1)
    # T, R = T[mask], R[mask]

    # Sort rows by increasing temperature; R rows must follow T
    # idx = np.argsort(T)
    # T, R = T[idx], R[idx]

    return T, nu, R


T, nu, R = load_field(96.2, IN)


# Plotting regular field, without sorting

nu_interest = np.argmin(np.abs(nu - 0.796))
R_interest = R[:, nu_interest]

plt.plot(T, R_interest, alpha = 0.8)
plt.xlim = 0
plt.ylim = 0

h_max = -1

# Also adding rough data 

rough = np.array([local_poly(R_interest, T, t, h_max, 1) for t in T])
plt.plot(T, rough, color = "red", alpha = 0.8)

# Also adding adaptive smoothed data

adaptive_smoothed = adaptive_smooth(R_interest, T)
plt.plot(T, adaptive_smoothed, color = "green", alpha = 0.8)



In [ ]:
# Test how fallback smoothing performs on all data 

from scripts.generate_linecut_plots import plot_all_linecuts
import os

FIELDS = [87, 96, 99, 103, 74, 87, 96.2, 151, 176]
OUT = Path("output/fallback_rough+adaptive_smoothing")
IN = Path(ROOT / "source_data")

if not os.path.exists(OUT):
    os.mkdir(OUT)


for field in FIELDS:
    plot_all_linecuts(field, 20, IN, OUT)


In [ ]:
# Test how normal fallback adaptive performs on all data 

from scripts.generate_linecut_plots import plot_all_linecuts
import os

FIELDS = [87, 96, 99, 103, 74, 87, 96.2, 151, 176]
OUT = Path("output/fallback_rough+fallback_adaptive_smoothing")
IN = Path(ROOT / "source_data")

if not os.path.exists(OUT):
    os.mkdir(OUT)


for field in FIELDS:
    plot_all_linecuts(field, 20, IN, OUT)
